<h1 align="center"><font color="red">Stateful vs Stateless Agent Design</font></h1>

<font color="pink">Senior Data Scientist.: Dr. Eddy Giusepe Chirinos Isidro</font>

# <font color="gree">Initial Setup</font>

You can find the API key in [groq](https://console.groq.com/home).

In [1]:

import os
from groq import Groq
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
 
# Get an API key in https://console.groq.com/keys and set it here:
GROQ_API_KEY = os.environ["GROQ_API_KEY"]
    
# Initializing the client:
client = Groq(api_key=GROQ_API_KEY)
 
# Using an efficient model from Groq: Llama 3.1 8B Instant
MODEL_ID = "llama-3.1-8b-instant"

An important setup decision here is the choice of a specific model. `llama-3.1-8b-instant` is highly cost-efficient model that is, at the time of writing, generously supported on `Groq's 2026 free tier`: it allows up to `14400 requests per day`. That makes it an ideal choice for illustrating the stateless and stateful agent paradigms bellow.

# <font color="blue">Stateless Agents: Fire and Forget</font>

Stateless agents treat each request as completely isolated  and independent. The agent reads the user prompt, invokes the LLM inference engine, and delivers the output. Once that execution cycle ends, everything is forgotten.

## <font color="gree">The Tradeoff</font>

Architectures based on `stateless agents` can be scaled horizontally with remarkable ease. Since no user memory is stored on a backend server, incoming requests can be forwarded to any available instance. There is, however, an important limitation in multi-turn conversations: the `frontend` must re-send the whole conversation history alongside every new request. As a result, the context window grows with a snowballing effect, quickly driving up token usage.

## <font color="gree">Illustrative Example</font>

This runnable code illustrates, through a basic scenario, how a stateless agent typically interacts with a `Groq` language model.

First, we define a `stateless_agent` function that emulates an agent’s interaction with our chosen model. Importantly, no state or memory of the conversation is kept internally. Instead, the previous conversation history can optionally be passed in as a parameter and appended to the current prompt. The API call to the Groq model takes place in `client.chat.completions.create()`.

In [2]:

def stateless_agent(prompt: str, provided_history: list = None) -> str:
    """
    The agent relies completely on the client to provide context.
    It retains no information from past interactions in local memory.
    """
    # Initializing with a system prompt
    messages = [{"role": "system",
                 "content": "You are a helpful, concise assistant."
                }
               ]
    
    # Appending whatever history the client provided:
    if provided_history:
        messages.extend(provided_history)
        
    # Appending the new prompt:
    messages.append({"role": "user",
                     "content": prompt
                    }
                   )
    
    # The LLM processes the entire chain of messages:
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        max_tokens=100
    )
    
    return response.choices[0].message.content.strip()

<font color="orange">To understand the limitation of a stateless agent, we simulate a simple user-model conversation through it:</font>

In [3]:
# --- Testing the Stateless Agent ---

print("--- Turn 1 ---")
prompt_1 = "Hi, my name is Alice and I am learning about API infrastructure."
response_1 = stateless_agent(prompt_1)
print(f"Agent: {response_1}")

print("\n--- Turn 2 (Without Client Context) ---")
# The agent fails here because it retained no memory of Turn 1
prompt_2 = "What is my name and what am I learning about?"
response_2 = stateless_agent(prompt_2)
print(f"Agent: {response_2}")

print("\n--- Turn 2 (With Client Context) ---")
# The frontend MUST inject the history into the payload for the agent to succeed
frontend_payload = [
    {"role": "user", "content": prompt_1},
    {"role": "assistant", "content": response_1}
]
response_3 = stateless_agent(prompt_2, provided_history=frontend_payload)
print(f"Agent: {response_3}")

--- Turn 1 ---
Agent: Nice to meet you, Alice. API infrastructure is a crucial part of modern software development. It's great that you're learning about it. What specific aspects of API infrastructure would you like to know or discuss? Would you like an overview, or would you like me to explain something in detail?

--- Turn 2 (Without Client Context) ---
Agent: I'm not able to retrieve personal information about you. We just started our conversation. How can I assist you today, or would you like to set up a learning topic?

--- Turn 2 (With Client Context) ---
Agent: Your name is Alice, and you're learning about API infrastructure.


The implementation es simple, but without a client or frontend that sends the conversation history to the agent on every turn, the agent's LLM lacks the context it needs to answer certain questions properly.

# <font color="blue">Stateful Agents: Context-driven Continuity</font>

Under this approach, the agent takes on the memory burden itself. The client, meanwhile, only needs to send the newest user prompt together with a unique identifier, normally associated with the current session. The agent then retrieves the session history or context from a database and appends the new message to it. Once the `LLM` inference has been processed, the agent updates the context in the database.

## <font color="gree">The Tradeoff</font>

This is a much neater experience from the client side. It also facilitates complex and asynchronous workflows in which agents may need to pause their execution and wait for `tools`, app responses, or `human approval`. But it all comes with a cost: scaling this solution becomes much harder, starting with the need for a persistent database layer in the architecture. In infrastructures that scale horizontally, strategies such as centralized memory caching with `Redis` may also become necessary to avoid `“localized amnesia”`, where a session’s history is stranded on the single instance that happened to serve the earlier turns.

## <font color="gree">Illustrative Example</font>

We illustrate the basic ideas behind a stateful agent by incorporating a “persistent” database layer. For simplicity, we use a tiny `SQLite database`. The key is to have the agent manage its own conversation memory instead of depending on a frontend to provide it externally:

In [ ]:
import sqlite3
import json

# Initializing an in-memory SQLite database for notebook testing:
conn = sqlite3.connect(':memory:') # Create interily on RAM
# conn = sqlite3.connect('agent_memory.db') # Persist on disk 
cursor = conn.cursor()
cursor.execute('''CREATE TABLE IF NOT EXISTS agent_memory (session_id TEXT PRIMARY KEY, history TEXT)''')
conn.commit()

def stateful_agent(session_id: str, new_prompt: str) -> str:
    """
    The agent manages its own state using a database.
    The client only sends the new prompt and their session ID.
    """
    # 1. Retrieving existing state from the database:
    cursor.execute("SELECT history FROM agent_memory WHERE session_id=?", (session_id,))
    row = cursor.fetchone()
    
    if row:
        conversation_history = json.loads(row[0])
    else:
        # Initializing with system prompt for new sessions
        conversation_history = [{"role": "system", "content": "You are a helpful, concise assistant."}]
        
    # 2. Appending the new user prompt:
    conversation_history.append({"role": "user", "content": new_prompt})
        
    # 3. Processing the LLM call using the retrieved history:
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=conversation_history,
        max_tokens=100
    ).choices[0].message.content.strip()
    
    # 4. Updating the state with the assistant's reply
    conversation_history.append({"role": "assistant", "content": response})
    
    # 5. Saving the new state back to the database
    cursor.execute('''
        INSERT INTO agent_memory (session_id, history) 
        VALUES (?, ?) 
        ON CONFLICT(session_id) DO UPDATE SET history=excluded.history
    ''', (session_id, json.dumps(conversation_history)))
    conn.commit()
    
    return response

Notice how the session identifier is used to query the relevant information from past interactions in the conversation at hand.

Now let’s try it all in a conversation similar to the previous one, but this time with the user asking the agent to recall the user’s own name:

In [5]:
# --- Testing the Stateful Agent ---

print("--- Turn 1 ---")
print(f"Agent: {stateful_agent('user_123', 'Hi, I am Eddy Giusepe and I want to scale my AI app.')}")

print("\n--- Turn 2 ---")
# Notice how the client NO LONGER sends the context payload. Just the session ID.
print(f"Agent: {stateful_agent('user_123', 'What was my name again?')}")

--- Turn 1 ---
Agent: Hello Eddy,

Scaling your AI app can be a complex process, but I'm here to help. To get started, could you please provide some more information about your app? 

Some questions to consider:

1. What kind of AI technology is your app utilizing? (e.g. machine learning, natural language processing, computer vision)
2. What type of data is your app working with? (e.g. images, text, audio)
3. What are some specific areas where you're

--- Turn 2 ---
Agent: Your name is Eddy Giusepe.


<font color="orange">This example is, of course, a long way from a scaled-up production architecture, but it serves to clarify the key difference between how stateful and stateless agents work.</font>

# <font color="blue">Wrapping Up: The Tradeoffs</font>

The choice between a `stateful` and a `stateless` architectural design boils down to properly matching the infrastructure to the workflow:

* `Stateless agents` are preferred in simple pipelines oriented to very specific tasks, like text extraction, summarization, or single-turn classification chatbots. They keep the architecture lightweight, which is usually enough in such use cases, avoiding database bottlenecks and allowing seamless horizontal scaling.

* `Stateful agents` make much more sense if we intend to develop long-running assistants, coding assistants, or multi-turn bots in applications like customer service. Because the agent owns the history, the client payload stays small on every turn, and the conversation can be trimmed or summarized server-side instead of being resent in full as it grows.